In [1]:
import os
import time
import pandas as pd

from pathlib import Path
from PIL import Image

env = os.environ.copy()
env['LD_LIBRARY_PATH'] = '/usr/local/lib:' + env.get('LD_LIBRARY_PATH', '')
env['PATH'] = '/usr/local/bin:' + env.get('PATH', '')

In [2]:
import subprocess

def compressImg(input_path, output_path, quality=0):
    """
    Compress an image using JPEG 2000 with specified quality (0 = lossless).
    
    Args:
        quality (int): Compression quality (0 = lossless, higher values = lossy).
    """
    # Build the command
    cmd = ['opj_compress', '-i', str(input_path), '-o', str(output_path), '-q', str(quality)]
    
    try:
        subprocess.run(cmd, check=True)
    except Exception as e:
        print(f'Error compressing {input_path}: {e}.')
        return False
    return True


def decompressImg(input_path, output_path):
    """
    Decompress a JPEG 2000 image.
    """
    cmd = ['opj_decompress', '-i', str(input_path), '-o', str(output_path)]
    try:
        subprocess.run(cmd, check=True)
    except Exception as e:
        print(f'Error decompressing {input_path}: {e}.')
        return False
    return True

In [3]:
def cal_compression_ratio(original_path, compressed_path):
    """
    Calculate compression rate based on file sizes.
    """
    original_size = os.path.getsize(original_path)
    compressed_size = os.path.getsize(compressed_path)

    with Image.open(original_path) as img:
        width, height = img.size
        npixels = width * height
    bpsp = (compressed_size * 8) / (npixels * 3)
    compression_ratio = compressed_size / original_size
    
    return compression_ratio, original_size, compressed_size, bpsp, width, height, npixels

In [4]:
def process_dataset(dataset_path, output_dir, mode="compress", cal_metric=True, verbose=False):
    """
    Process a dataset (Kodak or CLIC) using JPEG-2000, with optional compression rate calculation.
    """
    dataset_path = Path(dataset_path)
    output_dir   = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    imgs = list(dataset_path.glob('*.png'))
    if len(imgs) == 0:
        print(f'No Imgs found in {dataset_path}.')
        return
    
    rows = []
    for img in imgs:
        output_path = output_dir / f'{img.stem}.j2k' if mode == 'compress' else output_dir / f'{img.stem}_recon.png'
        
        start = time.time()
        if mode == 'compress':
            success = compressImg(img, output_path)
        elif mode == 'decompress':
            success = decompressImg(img, output_path)
        elapsed = time.time() - start   
          
        if success and cal_metric and mode == 'compress':
            compression_ratio, original_size, compressed_size, bpsp, width, height, npixels = cal_compression_ratio(img, output_path)
            if verbose:
                print(f'{img.name}: compression ratio = {compression_ratio:.2%} (Original: {original_size} bytes, Compressed: {compressed_size} bytes), bpsp: {bpsp}.')
            rows.append([img.name, bpsp, compression_ratio, original_size, compressed_size, width, height, npixels, elapsed])
            
    return rows

In [5]:
""" 1. compress Touch and Go (png) """
dataset_path = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/compressed/j2k'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/j2k_TouchandGoDataset-v2.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/j2k_TouchandGoDataset-v2.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,1400.000000,1400.000000,1400.000000,1400.000000,1400.0,1400.0,1400.0,1400.000000,1400.000000
mean,1.551758,0.616928,288034.526429,178762.547857,640.0,480.0,307200.0,0.061116,2.500300
std,0.332142,0.046721,44205.507025,38262.798833,0.0,0.0,0.0,0.008420,0.383728
min,0.874549,0.518611,192348.000000,100748.000000,640.0,480.0,307200.0,0.043011,1.669687
25%,1.335516,0.582372,253517.500000,153851.500000,640.0,480.0,307200.0,0.054977,2.200673
50%,1.460621,0.610627,276483.000000,168263.500000,640.0,480.0,307200.0,0.060153,2.400026
75%,1.622589,0.649134,310647.000000,186922.250000,640.0,480.0,307200.0,0.064716,2.696589
max,3.089401,0.775639,481097.000000,355899.000000,640.0,480.0,307200.0,0.098115,4.176189


In [6]:
""" 2. compress Objectfolder (png) """
dataset_path = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/compressed/j2k'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/j2k_ObjectFolder_1.0.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/j2k_ObjectFolder_1.0.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.0,20000.0,20000.0,20000.000000,20000.000000
mean,3.988553,1.006073,28541.906000,28717.579650,160.0,120.0,19200.0,0.014487,3.964154
std,0.382784,0.012976,2695.165442,2756.045559,0.0,0.0,0.0,0.002387,0.374329
min,3.329444,0.966118,24187.000000,23972.000000,160.0,120.0,19200.0,0.009756,3.359306
25%,3.699444,0.998742,26490.000000,26636.000000,160.0,120.0,19200.0,0.011958,3.679167
50%,3.863889,1.004469,27698.000000,27820.000000,160.0,120.0,19200.0,0.015096,3.846944
75%,4.232639,1.010937,30118.000000,30475.000000,160.0,120.0,19200.0,0.016188,4.183056
max,5.795000,1.119101,41962.000000,41724.000000,160.0,120.0,19200.0,0.028095,5.828056


In [7]:
""" 3. compress SSVTP (png) """
dataset_path = '/data/ssd/zhaoy/datasets/SSVTP/dataset-comp/test'
output_dir = '/data/ssd/zhaoy/datasets/SSVTP/compressed/j2k'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/j2k_SSVTP.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/j2k_SSVTP.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,918.000000,918.000000,918.000000,918.000000,918.0,918.0,918.0,918.000000,918.000000
mean,1.997093,0.896035,64301.678649,57516.281046,240.0,320.0,76800.0,0.021874,2.232697
std,0.104633,0.021478,4853.413152,3013.436596,0.0,0.0,0.0,0.004342,0.168521
min,1.764375,0.827461,53899.000000,50814.000000,240.0,320.0,76800.0,0.017068,1.871493
25%,1.908108,0.879945,60208.000000,54953.500000,240.0,320.0,76800.0,0.018465,2.090556
50%,1.986024,0.898185,63670.500000,57197.500000,240.0,320.0,76800.0,0.019408,2.210781
75%,2.075443,0.913042,67915.000000,59772.750000,240.0,320.0,76800.0,0.026716,2.358160
max,2.255625,0.955559,77399.000000,64962.000000,240.0,320.0,76800.0,0.039807,2.687465


In [8]:
""" 4. compress ObjTac (png) """
dataset_path = '/data/ssd/zhaoy/datasets/ObjTac/dataset-comp/train/image'
output_dir = '/data/ssd/zhaoy/datasets/ObjTac/compressed/j2k'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/j2k_ObjTac.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/j2k_ObjTac.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,51.000000,51.000000,51.000000,51.000000,51.0,51.000000,51.000000,51.000000,51.000000
mean,1.398914,2.682677,11784.803922,27611.666667,60.0,903.352941,54201.176471,0.053533,0.578609
std,0.699893,0.856922,7995.848535,16483.511901,0.0,391.917891,23515.073434,0.027836,0.328989
min,0.137141,1.684397,818.000000,1765.000000,60.0,331.000000,19860.000000,0.009284,0.063559
25%,0.731539,2.138162,4868.500000,13126.000000,60.0,587.500000,35250.000000,0.036303,0.253969
50%,1.504494,2.408985,10936.000000,25874.000000,60.0,813.000000,48780.000000,0.048103,0.629227
75%,1.861046,3.021340,19075.500000,44628.000000,60.0,1142.000000,68520.000000,0.068214,0.826285
max,2.697841,5.846734,28278.000000,59165.000000,60.0,1802.000000,108120.000000,0.135861,1.246181


In [10]:
""" 5. compress YCB-Slide (png) """
dataset_path = '/data/ssd/zhaoy/datasets/YCB-Slide/dataset-comp/train/image'
output_dir = '/data/ssd/zhaoy/datasets/YCB-Slide/compressed/j2k'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/j2k_YCB-Slide.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/j2k_YCB-Slide.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,25681.000000,25681.000000,25681.000000,25681.000000,25681.0,25681.0,25681.0,25681.000000,25681.000000
mean,1.916022,0.878281,62871.990966,55181.431097,240.0,320.0,76800.0,0.022489,2.183055
std,0.055410,0.014953,2765.646924,1595.797340,0.0,0.0,0.0,0.004353,0.096029
min,1.742674,0.828089,53941.000000,50189.000000,240.0,320.0,76800.0,0.016676,1.872951
25%,1.878437,0.868895,60956.000000,54099.000000,240.0,320.0,76800.0,0.018319,2.116528
50%,1.913472,0.879231,62700.000000,55108.000000,240.0,320.0,76800.0,0.019994,2.177083
75%,1.950312,0.889216,64532.000000,56169.000000,240.0,320.0,76800.0,0.026792,2.240694
max,2.153507,0.932055,73916.000000,62021.000000,240.0,320.0,76800.0,0.107598,2.566528
